In [ ]:
# ─── Environment Setup (do not edit) ────────────────────────────────────────
import os, sys
from pathlib import Path

# ── Colab: mount Drive and point to project root ──
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    # Set env file path for Colab before loading dotenv
    os.environ.setdefault("ENV_FILE", "/content/drive/MyDrive/Thesis_Final/fake-news-detection/.env.colab")
    # Add project root to sys.path
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
except ImportError:
    # Local / Vast.ai — project root is 2 levels up from notebooks/pipelines/
    PROJECT_ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

# ── Load .env ──
from dotenv import load_dotenv

env_file = os.environ.get("ENV_FILE") or PROJECT_ROOT / ".env"
if not Path(env_file).exists():
    raise FileNotFoundError(
        f".env file not found at: {env_file}\n"
        "Copy the correct .env.{{platform}}.example to .env and fill in your paths.\n"
        "Platforms: .env.vastai.example | .env.colab.example | .env.mac.example | .env.windows.example"
    )
load_dotenv(env_file, override=True)

# ── Resolve DATA_ROOT ──
from utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f"✅ Platform : {os.environ.get('PLATFORM', 'unknown')}")
print(f"✅ DATA_ROOT : {DATA_ROOT}")
print(f"✅ Exists   : {DATA_ROOT.exists()}")
# ─────────────────────────────────────────────────────────────────────────────

# Data Crawling — Vietnamese Fake News Detection

Automated, resumable crawling of Vietnamese news articles from the ViFactCheck dataset.

**Sources:** ViFactCheck HuggingFace dataset (`tranthaihoa/vifactcheck`)  
**Output:** Structured JSON files per split (train/dev/test) under `OUTPUT_DIR`  
**Resume:** Re-run this notebook at any time — already-crawled URLs are skipped automatically.

In [1]:
# ============================================================
# CONFIGURATION — edit this cell only
# ============================================================
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _env_file = Path(__file__).resolve().parents[2] / ".env" if "__file__" in dir() else Path.cwd().parents[1] / ".env"
    load_dotenv(_env_file, override=False)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

# DATA_ROOT: where large files (json, caches) are written.
# Defaults to DATA_ROOT env var, or PROJECT_ROOT if unset.
# To use an external drive, set DATA_ROOT in your .env before launching Jupyter.
DATA_ROOT = Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT

CONFIG = {
    # Data source
    "dataset_name": "tranthaihoa/vifactcheck",
    "url_column": "Url",
    "splits": ["train", "dev", "test"],
    "url_limit": None,           # None = all URLs; set int to limit (e.g. 100 for testing)

    # Output
    "output_dir": DATA_ROOT / "data" / "json",
    "cache_dir": DATA_ROOT / "data" / "caches",
    "output_format": "ml_training",  # options: custom, ml_training, research, detailed

    # Crawl settings
    "max_concurrent": 3,        # parallel requests per split
    "save_interval": 50,         # checkpoint every N completed URLs
    "retry_failed": True,       # set True to retry previously failed URLs
    "domain_concurrency": {      # per-domain concurrency overrides (None = use defaults)
        "dantri.com.vn": 1,      # strict serialization to avoid 429s
    },
}

In [2]:
import subprocess, sys

_PACKAGES = ["loguru", "tqdm", "beautifulsoup4", "lxml", "httpx",
             "nest-asyncio", "datasets", "Pillow"]

for pkg in _PACKAGES:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"Installed {pkg}")
print("Dependencies ready.")

Installed beautifulsoup4


/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Installed Pillow
Dependencies ready.


In [3]:
import sys, os
import nest_asyncio
nest_asyncio.apply()

# Add project root to path
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Create output directories
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

from src.crawler.crawler_factory import CrawlerFactory
from src.processing.dataset_handler import DatasetHandler

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {CONFIG['output_dir']}")
print(f"Cache dir:    {CONFIG['cache_dir']}")

Project root: /Users/haila/Desktop/personal-app/CascadeProjects/fake-new-detection
Output dir:   /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/data/json
Cache dir:    /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/data/caches


## Step 1: Load URLs from ViFactCheck Dataset

In [4]:
dataset_handler = DatasetHandler(CONFIG["dataset_name"])

split_urls = {}
for split in CONFIG["splits"]:
    urls = dataset_handler.get_urls_from_split(
        split=split,
        url_column=CONFIG["url_column"],
        limit=CONFIG["url_limit"],
    )
    split_urls[split] = urls
    print(f"  {split:6s}: {len(urls):,} URLs")

total = sum(len(v) for v in split_urls.values())
print(f"\nTotal URLs to process: {total:,}")

--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 5062 URLs to crawl. ---
  train : 5,062 URLs
--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 723 URLs to crawl. ---
  dev   : 723 URLs
--- Loading dataset 'tranthaihoa/vifactcheck' from Hugging Face... ---
--- Extracting URLs from the 'Url' column... ---
--- Found 1447 URLs to crawl. ---
  test  : 1,447 URLs

Total URLs to process: 7,232


## Step 2: Check Resume State

In [5]:
import json

for split in CONFIG["splits"]:
    cache_file = CONFIG["cache_dir"] / f"crawling_status_{split}.json"
    failed_file = CONFIG["cache_dir"] / f"failed_urls_{split}.json"

    completed = 0
    failed = 0

    if cache_file.exists():
        with open(cache_file) as f:
            data = json.load(f)
            completed = data.get("length", len(data.get("urls", [])))

    if failed_file.exists():
        with open(failed_file) as f:
            failed = len(json.load(f))

    total = len(split_urls[split])
    remaining = total - completed
    print(f"  {split:6s}: {completed:,} done / {failed:,} failed / {remaining:,} remaining")

print("\nRe-run this notebook to resume. Already-crawled URLs are skipped automatically.")

  train : 0 done / 0 failed / 5,062 remaining
  dev   : 0 done / 0 failed / 723 remaining
  test  : 0 done / 0 failed / 1,447 remaining

Re-run this notebook to resume. Already-crawled URLs are skipped automatically.


## Step 3: Crawl Articles

Progress is shown per split. Checkpoints are saved every `save_interval` URLs.
Re-run this cell at any time to resume from where crawling stopped.

In [6]:
async def crawl_split(split: str, urls: list) -> dict:
    """Crawl one split, return summary dict."""
    cache_file = str(CONFIG["cache_dir"] / f"crawling_status_{split}.json")
    failed_file = str(CONFIG["cache_dir"] / f"failed_urls_{split}.json")
    output_file = f"news_data_vifactcheck_{split}.json"

    print(f"\n{'='*50}")
    print(f"Crawling split: {split} ({len(urls):,} URLs)")
    print(f"{'='*50}")

    factory = CrawlerFactory(
        cache_filename=cache_file,
        failed_log_filename=failed_file,
    )

    await factory.crawl_and_save_all(
        urls=urls,
        output_filename=output_file,
        format_name=CONFIG["output_format"],
        max_concurrent=CONFIG["max_concurrent"],
        retry_failed=CONFIG["retry_failed"],
        save_interval=CONFIG["save_interval"],
        output_dir=str(CONFIG["output_dir"]),
        domain_concurrency=CONFIG.get("domain_concurrency"),
    )

    # Return summary
    completed = 0
    if Path(cache_file).exists():
        with open(cache_file) as f:
            data = json.load(f)
            completed = data.get("length", 0)

    failed = 0
    if Path(failed_file).exists():
        with open(failed_file) as f:
            failed = len(json.load(f))

    return {"split": split, "total": len(urls), "completed": completed, "failed": failed}

In [7]:
import asyncio

summaries = []
for split in CONFIG["splits"]:
    summary = await crawl_split(split, split_urls[split])
    summaries.append(summary)

print("\nAll splits processed.")


Crawling split: train (5,062 URLs)
2026-06-07 17:33:53 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:249 - Retrying 0 previously failed URLs.
2026-06-07 17:33:53 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:252 - URLs to crawl: 5062 (skipped: 0, concurrency=3)


Crawling:   0%|          | 2/5062 [00:02<1:59:28,  1.42s/it]

Directory /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/data/jpg/bao_chinh_phu created.


Crawling:   0%|          | 3/5062 [00:03<1:27:37,  1.04s/it]

Directory /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/data/jpg/bao_tin_tuc created.


...

2026-06-07 18:52:54 | WARNING  | helpers.httpx_client:_request:110 - Request error for URL('https://tuoitre.vn/my-quyet-liet-chinh-phuc-lai-chau-phi-do-so-nga-va-trung-quoc-20230328164249345.htm'): Server disconnected without sending a response.. Retrying in 0.5s...


Crawling:  97%|█████████▋| 1401/1447 [16:48<01:20,  1.75s/it]

2026-06-07 18:54:14 | WARNING  | helpers.httpx_client:_request:110 - Request error for URL('https://tuoitre.vn/le-thuy-kim-tu-long-tien-biet-nghe-si-vu-linh-20230306234524566.htm'): . Retrying in 0.5s...


Crawling: 100%|██████████| 1447/1447 [17:41<00:00,  1.36it/s]

2026-06-07 18:55:07 | INFO     | src.crawler.crawler_factory:crawl_and_save_all:344 - Saved 1447 new + 0 existing = 1447 total articles to /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/data/json/news_data_vifactcheck_test.json
2026-06-07 18:55:07 | INFO     | src.crawler.crawler_factory:_print_summary:356 - 
--- Crawling Summary ---
2026-06-07 18:55:07 | INFO     | src.crawler.crawler_factory:_print_summary:357 - Completed (total): 758 URLs
2026-06-07 18:55:07 | INFO     | src.crawler.crawler_factory:_print_summary:358 - Failed this run:   0 URLs
2026-06-07 18:55:07 | INFO     | src.crawler.crawler_factory:_print_summary:359 - Failed (total):    0 URLs

All splits processed.


## Step 4: Results Summary

In [8]:
print(f"\n{'Split':<8} {'Total':>8} {'Completed':>10} {'Failed':>8} {'Rate':>8}")
print("-" * 50)

for s in summaries:
    rate = f"{s['completed'] / s['total'] * 100:.1f}%" if s['total'] else "—"
    print(f"{s['split']:<8} {s['total']:>8,} {s['completed']:>10,} {s['failed']:>8,} {rate:>8}")

print("-" * 50)
total_urls = sum(s["total"] for s in summaries)
total_done = sum(s["completed"] for s in summaries)
total_fail = sum(s["failed"] for s in summaries)
overall_rate = f"{total_done / total_urls * 100:.1f}%" if total_urls else "—"
print(f"{'TOTAL':<8} {total_urls:>8,} {total_done:>10,} {total_fail:>8,} {overall_rate:>8}")


Split       Total  Completed   Failed     Rate
--------------------------------------------------
train       5,062      5,062        0   100.0%
dev           723        723        0   100.0%
test        1,447      1,447        0   100.0%
--------------------------------------------------
TOTAL       7,232      7,232        0   100.0%


In [9]:
print("\nOutput files:")
for split in CONFIG["splits"]:
    out_file = CONFIG["output_dir"] / f"news_data_vifactcheck_{split}.json"
    if out_file.exists():
        with open(out_file) as f:
            count = len(json.load(f))
        size_mb = out_file.stat().st_size / 1024 / 1024
        print(f"  {split:6s}: {out_file.name} — {count:,} articles ({size_mb:.1f} MB)")
    else:
        print(f"  {split:6s}: not yet created")

print("\nNext step: Run notebooks/02_preprocessing.ipynb")


Output files:
  train : news_data_vifactcheck_train.json — 5,062 articles (14.9 MB)
  dev   : news_data_vifactcheck_dev.json — 723 articles (2.1 MB)
  test  : news_data_vifactcheck_test.json — 1,447 articles (4.4 MB)

Next step: Run notebooks/02_preprocessing.ipynb
